# Run the saved fraud and activity models
Run **Setup**, then either dataset section. Each section compares the baseline and best prediction for the same input row. The defaults use examples from the supplied training files to demonstrate inference. For new data, change the filenames at the top of the relevant section.

**Local Jupyter:** open this notebook from the project or place the needed pickles and datasets beside it. **Google Colab:** upload this notebook, the two pickles for your chosen task, and its data files to the runtime's Files panel. Load only trusted pickles. The exports were created with Python 3.12.

In [1]:
# Setup: if dependencies are missing or incompatible, uncomment this line, run it, then restart the kernel.
# %pip install numpy==2.5.5 pandas==3.0.1 scipy==1.18.1 scikit-learn==1.9.1 imbalanced-learn==0.14.2 xgboost==3.4.1 lightgbm==4.7.0 cloudpickle==3.1.3 threadpoolctl==3.8.0
import pickle
import zipfile
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()  # Change to the project folder or the directory containing uploaded files.

def locate(name):
    path = Path(name)
    if (ROOT / path).is_file():
        return ROOT / path
    for root in (ROOT, ROOT.parent):
        for folder in ("baseline_models", "best_models", "Code Prototypes", "Docs and Data"):
            matches = list((root / folder).rglob(path.name))
            if len(matches) == 1:
                return matches[0]
            if len(matches) > 1:
                raise ValueError(f"Multiple copies of {name}. Supply an explicit path.")
    raise FileNotFoundError(f"Place {name} beside the notebook or supply its full path.")

def load_model(name):
    with locate(name).open("rb") as stream:
        return pickle.load(stream)

## 1 · Credit card fraud
Output labels are **0 = legitimate** and **1 = fraud**. The default scores the final 20 transactions in chronological order, with the preceding transactions as history. For your own batch, set `HISTORY_FILE` to the earlier transaction file and `NEW_PAYMENTS` to the new CSV or ZIP. Use the original column names. Each transaction should appear once across the two inputs. Output rows follow batch order.

In [2]:
HISTORY_FILE = "cct_train.zip"
NEW_PAYMENTS = None  # Example: "cct_test.csv". None runs the built-in example.

fraud_best = load_model("fraud_best_random_forest_transaction_history.pkl")
fraud_baseline = load_model("fraud_baseline_random_forest_amount_hour.pkl")

def read_payments(filename):
    # Read just the model inputs; keep identifiers as strings to preserve leading zeros.
    ids = ["cc_num", "acct_num", "first", "last", "dob", "merchant", "category"]
    options = dict(usecols=fraud_best.fields, dtype={key: "string" for key in ids})
    path = locate(filename)
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as archive:
            members = [s for s in archive.namelist() if s.lower().endswith(".csv") and "__MACOSX/" not in s]
            if len(members) != 1:
                raise ValueError("The payments ZIP must contain one CSV file.")
            with archive.open(members[0]) as stream:
                return pd.read_csv(stream, **options)
    return pd.read_csv(path, **options)

history = read_payments(HISTORY_FILE)
if NEW_PAYMENTS is None:
    ordered = history.sort_values("unix_time", kind="stable")
    history, payments = ordered.iloc[:-20], ordered.iloc[-20:]
else:
    payments = read_payments(NEW_PAYMENTS)

fraud_predictions = payments[["trans_date", "trans_time", "amt"]].copy()
fraud_predictions["baseline_prediction"] = fraud_baseline.predict(payments)
fraud_predictions["best_prediction"] = fraud_best.predict(payments, history)
print(f"Scored {len(fraud_predictions):,} transactions. First five:")
print(fraud_predictions.head().to_string(index=False))
# Optional: fraud_predictions.to_csv("fraud_predictions.csv", index=False)

Scored 20 transactions. First five:
trans_date trans_time   amt  baseline_prediction  best_prediction
2023-01-10   23:30:44 52.18                    0                0
2023-01-10   23:34:12 62.99                    0                0
2023-01-10   23:35:43 10.89                    0                0
2023-01-10   23:37:02  8.27                    0                0
2023-01-10   23:37:39  1.07                    0                0


## 2 · Activity recognition
Inputs are **328 sensor columns** in training order, in a headerless CSV. The reader also accepts the supplied training format with a 329th activity-label column. Predictions are activity names. Set `ACTIVITY_FILE` to your own CSV and `ROW_LIMIT = None` to score every reading.

In [3]:
ACTIVITY_FILE = "har_train.csv"
ROW_LIMIT = 20  # Set to None for the entire file.

activity_baseline = load_model("activity_baseline_xgboost_lightgbm.pkl")
activity_best = load_model("activity_best_15_neighbor_sports_booster_ensemble.pkl")
readings = pd.read_csv(locate(ACTIVITY_FILE), header=None, nrows=ROW_LIMIT)
if readings.shape[1] not in (328, 329):
    raise ValueError(f"Expected 328 sensors and an optional label; found {readings.shape[1]} columns.")
sensors = readings.iloc[:, :328].apply(pd.to_numeric, errors="raise")
activity_predictions = pd.DataFrame({
    "baseline_prediction": activity_baseline.predict(sensors),
    "best_prediction": activity_best.predict(sensors),
}, index=readings.index)
if readings.shape[1] == 329:
    activity_predictions["provided_label"] = readings.iloc[:, -1]
print(f"Scored {len(activity_predictions):,} readings. First five:")
print(activity_predictions.head().to_string())
# Optional: activity_predictions.to_csv("activity_predictions.csv", index_label="row")

Scored 20 readings. First five:
  baseline_prediction best_prediction provided_label
0              Tennis          Tennis         Tennis
1             Sitting         Sitting        Sitting
2             Cycling         Cycling        Cycling
3              Tennis          Tennis         Tennis
4            Football        Football       Football
